# CSE 251B FineWeb-Edu Pretraining

1. Check the GPU with `nvidia-smi`.
2. Mount Google Drive.
3. Prepare or reuse FineWeb-Edu shards under `MyDrive/251B/dataset/fineweb_edu/`.
4. Train using `config/train_gpt2.py`.
5. Save the checkpoint and full training log under `MyDrive/251B/results/<RUN_NAME>/`.


Note: quick guidance on vscode:
1. install colab extension: google.colab
2. click Detecting Kernels
3. choose colab
4. new colab server
5. gpu A100 High Ram Latest (choose cpu if u haven't downloaded the datasets)
6. choose python 3 ipykernel
7. run all the cells

## 0. Check GPU and wandb

In [ ]:
!nvidia-smi

if you want to login wandb

In [ ]:
import wandb
wandb.login()

## 1. Mount Drive And Define Paths


Drive layout used by this notebook:

- Dataset cache: `MyDrive/251B/dataset/fineweb_edu/`
- Results: `MyDrive/251B/results/`
- Local repo: `/content/milestone`

Note: Running this cell will open a webpage to ask for your permissions.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os
import shutil
import subprocess
import time

DRIVE_ROOT = Path('/content/drive/MyDrive/251B')
DRIVE_DATASET_DIR = DRIVE_ROOT / 'dataset' / 'fineweb_edu'
DRIVE_RESULTS_DIR = DRIVE_ROOT / 'results'

REPO_DIR = Path('/content/milestone')
LOCAL_DATASET_DIR = REPO_DIR / 'data' / 'fineweb_edu'

DRIVE_DATASET_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print('Drive root:       ', DRIVE_ROOT)
print('Drive dataset:    ', DRIVE_DATASET_DIR)
print('Drive results dir:', DRIVE_RESULTS_DIR)
print('Local repo:       ', REPO_DIR)
print('Local dataset:    ', LOCAL_DATASET_DIR)

## 2. Prepare The Repository

In [ ]:
REPO_URL = 'https://github.com/FufenNan/milestone.git'

if not (REPO_DIR / 'train.py').exists():
    assert REPO_URL, 'Set REPO_URL before cloning the repo into /content/milestone.'
    print(f'Cloning repo into {REPO_DIR} ...')
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

assert (REPO_DIR / 'train.py').exists(), (
    'Could not find train.py. Put the repo at /content/milestone, '
    'or set REPO_URL above to clone it into Colab.'
)

os.chdir(REPO_DIR)
print('Using repo:', REPO_DIR)

## 3. Install Dependencies

In [ ]:
!pip install -q datasets tiktoken tqdm wandb

## 4. Prepare FineWeb-Edu Dataset

This cell makes one decision:

- If `MyDrive/251B/dataset/fineweb_edu/` already has enough shards, copy them to `/content/milestone/data/fineweb_edu/`.
- Otherwise, download/tokenize FineWeb-Edu locally, then copy the prepared shards back to Drive.

In [ ]:
SETUP_LOG = DRIVE_RESULTS_DIR / 'dataset_prepare.log'

def log(msg):
    line = f'[{time.strftime("%Y-%m-%d %H:%M:%S")}] {msg}'
    print(line)
    with open(SETUP_LOG, 'a', encoding='utf-8') as f:
        f.write(line + '\n')

def count_shards(path):
    path = Path(path)
    train = sorted(path.glob('fineweb_train_*.bin'))
    val = sorted(path.glob('fineweb_val_*.bin'))
    return len(train), len(val)

def shard_bytes(path):
    path = Path(path)
    return sum(p.stat().st_size for p in path.glob('fineweb_*.bin'))

def has_ready_dataset(path, min_train_shards):
    train_count, val_count = count_shards(path)
    return train_count >= min_train_shards and val_count >= 1

def sync_dir(src, dst):
    src = Path(src)
    dst = Path(dst)
    dst.mkdir(parents=True, exist_ok=True)
    try:
        subprocess.run(['rsync', '-ah', '--info=progress2', f'{src}/', f'{dst}/'], check=True)
    except FileNotFoundError:
        shutil.copytree(src, dst, dirs_exist_ok=True)

SHARD_SIZE = 100_000_000
MAX_SHARDS = 22  # 1 val shard + 21 train shards; train side is about 2.1B tokens
MIN_TRAIN_SHARDS = MAX_SHARDS - 1
NUM_PROC = max(1, (os.cpu_count() or 2) // 2)

LOCAL_DATASET_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_DATASET_DIR.mkdir(parents=True, exist_ok=True)

log('Checking FineWeb-Edu dataset on Drive...')
drive_train, drive_val = count_shards(DRIVE_DATASET_DIR)
drive_gb = shard_bytes(DRIVE_DATASET_DIR) / 1e9
DATA_READY_ON_DRIVE = has_ready_dataset(DRIVE_DATASET_DIR, MIN_TRAIN_SHARDS)
log(f'Drive shards: train={drive_train}, val={drive_val}, size={drive_gb:.2f} GB, ready={DATA_READY_ON_DRIVE}')

if DATA_READY_ON_DRIVE:
    log('Drive dataset is ready. Copying shards to local Colab runtime...')
    sync_dir(DRIVE_DATASET_DIR, LOCAL_DATASET_DIR)
else:
    log('Drive dataset is not ready. Downloading/tokenizing FineWeb-Edu locally...')
    cmd = [
        'python', '-u', 'data/fineweb_edu/prepare.py',
        '--data-root', str(LOCAL_DATASET_DIR),
        '--streaming',
        '--shard-size', str(SHARD_SIZE),
        '--max-shards', str(MAX_SHARDS),
        '--num-proc', str(NUM_PROC),
    ]
    log('Running: ' + ' '.join(cmd))
    with open(SETUP_LOG, 'a', encoding='utf-8') as setup_log:
        proc = subprocess.Popen(
            cmd,
            cwd=REPO_DIR,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        for line in proc.stdout:
            print(line, end='')
            setup_log.write(line)
            setup_log.flush()
        ret = proc.wait()
    if ret != 0:
        raise RuntimeError(f'FineWeb-Edu preparation failed with exit code {ret}. See {SETUP_LOG}')

    log('Copying prepared FineWeb-Edu shards back to Drive...')
    sync_dir(LOCAL_DATASET_DIR, DRIVE_DATASET_DIR)

local_train, local_val = count_shards(LOCAL_DATASET_DIR)
local_gb = shard_bytes(LOCAL_DATASET_DIR) / 1e9
log(f'Local shards: train={local_train}, val={local_val}, size={local_gb:.2f} GB')
assert has_ready_dataset(LOCAL_DATASET_DIR, MIN_TRAIN_SHARDS), 'FineWeb-Edu dataset is not ready locally.'
log('FineWeb-Edu dataset is ready for training.')
print('Setup log:', SETUP_LOG)

## 5. Train

The training command uses `python -u` and redirects stderr into stdout. 

After training completes, the notebook copies `ckpt.pt` and `train.log` to `MyDrive/251B/results/<RUN_NAME>/`.

In [ ]:
RUN_NAME = 'fineweb_edu_nano_' + time.strftime('%m%d_%H%M%S')
LOCAL_RUN_DIR = REPO_DIR / 'output' / RUN_NAME
DRIVE_RUN_DIR = DRIVE_RESULTS_DIR / RUN_NAME
LOCAL_LOG = REPO_DIR / f'{RUN_NAME}.log'

env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'

cmd = [
    'python', '-u', 'train.py', 'config/train_gpt2.py',
    f'--out_dir={RUN_NAME}',
    f'--wandb_run_name={RUN_NAME}',
]

print('Run name:     ', RUN_NAME)
print('Command:      ', ' '.join(cmd))
print('Local output: ', LOCAL_RUN_DIR)
print('Drive output: ', DRIVE_RUN_DIR)
print('Local log:    ', LOCAL_LOG)

with open(LOCAL_LOG, 'w', encoding='utf-8') as log_file:
    proc = subprocess.Popen(
        cmd,
        cwd=REPO_DIR,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in proc.stdout:
        print(line, end='')
        log_file.write(line)
        log_file.flush()
    ret = proc.wait()

if ret != 0:
    raise RuntimeError(f'Training failed with exit code {ret}. See {LOCAL_LOG}')

assert (LOCAL_RUN_DIR / 'ckpt.pt').exists(), 'Training finished but ckpt.pt was not found.'
DRIVE_RUN_DIR.mkdir(parents=True, exist_ok=True)
sync_dir(LOCAL_RUN_DIR, DRIVE_RUN_DIR)
shutil.copy2(LOCAL_LOG, DRIVE_RUN_DIR / 'train.log')
print('Saved model artifacts to:', DRIVE_RUN_DIR)
print('Saved training log to:   ', DRIVE_RUN_DIR / 'train.log')